# Phase 6 submission

Prep v2 + six slots + per-diagnosis attention, five folds, target `steven_v4`.

Three things differ from `phase5-submit`, all of them consequences of the artifact change:

- **`prep_slots`, not `prep_study`.** Test studies must reach the model through the same
  130mm physical-scale crop, six-slot layout and contiguous 3-slice groups the training
  artifacts were built with. Prepping at 224 directly would be faster and wrong: training
  saw 336 stored then resized to 224, and that resampling chain is part of what the model
  learned.
- **The loader returns a presence mask**, and it is passed to the model. Dropping it would
  make every absent slot read as real black anatomy, which is exactly the bias the masked
  pooling exists to remove -- and it would show up only as a slightly worse score.
- **`pretrained=False`.** Internet is off and every weight is overwritten by the checkpoint
  anyway; leaving it True would try to reach the network and fail the submission outright.

The 0.5 fallback is unchanged and remains non-negotiable: a submission that raises on one
hidden study scores zero.


In [ ]:
import glob, os, shutil, sys, tempfile, time

GIT_SHA = 'phase6-submit'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]
CKPTS = sorted(glob.glob('/kaggle/input/**/c2_6slot_attn_fold*.pt', recursive=True))
assert len(CKPTS) == 5, f'expected 5 fold checkpoints, found {len(CKPTS)}: {CKPTS}'

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

# A stale src here does not crash -- it silently prepares different pixels from the
# ones the checkpoints were trained on, and the only symptom is a lower score.
import inspect
from knee.dicom import SLOTS, select_slots, select_slice_groups
from knee.model import KneeModel
from knee.prep import crop_to_mm, prep_slots
assert 'head' in inspect.signature(KneeModel.__init__).parameters, 'src predates the c2 head'
assert len(SLOTS) == 6, 'src slot table is not the six-slot layout'
print('src carries the Phase 6 prep and head |', len(CKPTS), 'checkpoints')

In [ ]:
import numpy as np
import pandas as pd
import torch

from knee.dataset import PreppedSlotDataset
from knee.infer import LABEL_COLUMNS, build_submission
from knee.prep import save_study_npz

# Exactly the training artifact's prep parameters and the confirmed config's read
# parameters. These are the numbers that must not drift from the confirm kernel.
PREP_GROUPS, PREP_CROP_MM, PREP_SIZE = 5, 130.0, 336
READ_SIZE, HEAD, BACKBONE = 224, 'slot_attention', 'efficientnet_b0'
SLOT_NAMES = [n for n, _, _ in SLOTS]

test_df = pd.read_csv(f'{COMP_DIR}/test.csv')
test_series_df = pd.read_csv(f'{COMP_DIR}/test_series.csv')
TEST_ROOT = f'{COMP_DIR}/test_series'
print(f'{len(test_df)} test studies')

# Probe rather than assert: a hard GPU requirement turns an odd accelerator
# assignment into a zero on the scored run.
device = 'cpu'
if torch.cuda.is_available():
    try:
        _ = torch.zeros(1, device='cuda') + 1
        device = 'cuda'
    except Exception as exc:
        print(f'GPU present but unusable: {exc}')
print('device:', device)

models = []
for path in CKPTS:
    m = KneeModel(backbone_name=BACKBONE, num_labels=len(LABEL_COLUMNS),
                  pretrained=False, head=HEAD)
    m.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    models.append(m.to(device).eval())
print(f'{len(models)} fold models loaded')

In [ ]:
NPZ_DIR = tempfile.mkdtemp(prefix='knee_prep_')
prep_seconds, forward_seconds = [], []
lat_routes, slot_fill = {}, {}

def predict_one(study_uid):
    """prep_slots -> npz -> the tested loader -> probability mean of 5 fold models.
    Anything raising in here becomes a row of 0.5 via build_submission."""
    t0 = time.time()
    slot_slices, meta = prep_slots(
        study_uid, TEST_ROOT, test_series_df,
        n_groups=PREP_GROUPS, crop_mm=PREP_CROP_MM, out_size=PREP_SIZE)
    npz_path = os.path.join(NPZ_DIR, f'{study_uid}.npz')
    save_study_npz(npz_path, slot_slices, meta)
    prep_seconds.append(time.time() - t0)
    lat_routes[meta.get('route')] = lat_routes.get(meta.get('route'), 0) + 1
    slot_fill[len(slot_slices)] = slot_fill.get(len(slot_slices), 0) + 1

    try:
        dataset = PreppedSlotDataset([study_uid], NPZ_DIR, n_groups=PREP_GROUPS,
                                     out_size=READ_SIZE, slots=SLOT_NAMES)
        image, mask, _, _ = dataset[0]
        t1 = time.time()
        batch = image.unsqueeze(0).to(device)
        mask_batch = mask.unsqueeze(0).to(device)
        with torch.no_grad():
            # probability mean, not rank mean: measured 2026-09-07, prob-mean won
            # 11 of 11 combinations on homogeneous members (NOTES, infer.rank_mean)
            probs = np.mean([torch.sigmoid(m(batch, mask=mask_batch))[0].cpu().numpy()
                             for m in models], axis=0)
        forward_seconds.append(time.time() - t1)
        return probs
    finally:
        os.remove(npz_path)

t0 = time.time()
submission = build_submission(test_df['StudyInstanceUID'].astype(str).tolist(), predict_one)
elapsed = time.time() - t0
print(f'{elapsed / 60:.1f} min for {len(submission)} studies '
      f'({elapsed / max(len(submission), 1):.2f} s/study)')
print(f'prep    {np.mean(prep_seconds):.3f}s/study (p95 {np.percentile(prep_seconds, 95):.3f})')
print(f'forward {np.mean(forward_seconds):.3f}s/study for {len(models)} models')
print(f'laterality routes: {lat_routes}')
print(f'slots filled per study: {dict(sorted(slot_fill.items()))}')
fallbacks = int((submission[LABEL_COLUMNS] == 0.5).all(axis=1).sum())
print(f'0.5 fallbacks: {fallbacks} of {len(submission)}')

In [ ]:
submission.to_csv('submission.csv', index=False)
print(submission.head())

assert len(submission) == len(test_df), 'row count must match test studies exactly'
assert list(submission.columns) == ['StudyInstanceUID'] + LABEL_COLUMNS, 'column contract'
assert submission[LABEL_COLUMNS].isna().sum().sum() == 0, 'no NaNs allowed'
vals = submission[LABEL_COLUMNS].to_numpy()
assert (vals >= 0).all() and (vals <= 1).all(), 'probabilities must be in [0, 1]'

# Predictions should track the training positive rates and preserve their ordering.
# A wild departure means something broke upstream of scoring, which no assert above
# would catch.
print(f'\n{"label":20s} {"mean pred":>10s}')
for label in LABEL_COLUMNS:
    print(f'{label:20s} {submission[label].mean():10.4f}')